# Tái lập thực nghiệm: Khai thác tập phổ biến bằng thuật toán Eclat (Julia)

Tài liệu thực thi này trình bày toàn bộ quy trình kiểm chứng tính đúng đắn, đánh giá hiệu năng và tái lập kết quả ứng dụng của gói `EclatFIM` trên môi trường Google Colab.

- **Thuật toán:** Eclat (M. J. Zaki, *IEEE Transactions on Knowledge and Data Engineering*, vol. 12, no. 3, pp. 372–390, 2000).
- **Nhóm thực hiện:** Phạm Trần Anh Quân (MSSV: 24127226 — Thuật toán, hiệu năng và Ứng dụng 2), Nguyễn Tiến Cường (MSSV: 24127337 — I/O, kiểm chứng, hạ tầng thực nghiệm và Ứng dụng 1).
- **Chế độ thực thi:** Biến `CHE_DO` xác định phạm vi tái lập giữa chế độ `"nhanh"` (tập con lưới ngưỡng, tổng thời gian trong giới hạn phiên Colab miễn phí) và chế độ `"day_du"` (toàn bộ lưới thực nghiệm).

Mỗi mục được thiết kế độc lập về mặt thực thi nhằm bảo đảm khả năng tái lập và kiểm chứng từng thành phần.

In [ ]:
# Tham số cấu hình tái lập
import os
CHE_DO = "nhanh"      # Giá trị hợp lệ: "nhanh" hoặc "day_du"
REPO_URL = "https://github.com/24127226/16-FIM-DATA-MINING.git"
REPO_BRANCH = "main"  # Nhánh hoặc nhãn phát hành (tag) dùng để tái lập
REPO_DIR = "/content/16-FIM-DATA-MINING"
JULIA_VERSION = "1.11.4"
os.environ["FIM_DATA_BASE"] = "https://huggingface.co/datasets/cuongnguyen0794/16-fim-data-mining/resolve/main"

print(f"Chế độ thực thi: {CHE_DO} | Kho mã nguồn: {REPO_URL} ({REPO_BRANCH})")

## Phần 1. Thiết lập môi trường thực thi (Julia 1.11.4, Java và SPMF)

Khởi tạo bản sao kho mã nguồn, cài đặt trình biên dịch Julia phiên bản `1.11.4`, khôi phục các gói phụ thuộc theo `Manifest.toml`, kiểm tra môi trường Java và tải cài đặt tham chiếu `spmf.jar` kèm xác minh mã băm SHA-256.

In [ ]:
import os, shutil, subprocess

# Sao chép kho mã nguồn
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# Thiết lập Julia 1.11.4
julia_bin = f"/usr/local/julia-{JULIA_VERSION}/bin/julia"
if not os.path.exists(julia_bin) and shutil.which("julia") is None:
    url = f"https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
    subprocess.run(f"curl -fsSL {url} | tar -xz -C /usr/local", shell=True, check=True)
    subprocess.run(["ln", "-sf", julia_bin, "/usr/local/bin/julia"], check=True)

!julia --version
!julia --project=. -e "using Pkg; Pkg.instantiate(); Pkg.precompile()"

In [ ]:
# Kiểm tra phiên bản Java, tải spmf.jar và ghi nhận cấu hình phần cứng
!java -version || true
!julia --project=. scripts/fetch_spmf.jl
!julia --project=. scripts/env_info.jl

## Phần 2. Thu thập dữ liệu và kiểm tra toàn vẹn SHA-256

Tải các bộ dữ liệu benchmark (`chess`, `mushroom`, `retail`, `accidents`, `T10I4D100K`) từ kho dữ liệu công khai trên Hugging Face (không yêu cầu xác thực) và kiểm tra mã băm SHA-256 đối chiếu với bảng đặc trưng trong đề bài.

In [ ]:
import os
os.chdir(REPO_DIR)
!julia --project=. scripts/fetch_data.jl benchmark

## Phần 3. Kiểm chứng giao diện dòng lệnh trên các tập dữ liệu chuẩn

Thực thi giao diện dòng lệnh `demo.jl` trên hai tập dữ liệu chuẩn: `pasquier99.txt` tại ngưỡng `minsup = 40%` (kỳ vọng 15 tập phổ biến) và `bookstore.txt` tại ngưỡng `minsup = 0.5` (kỳ vọng 19 tập phổ biến, kèm bộ đếm nội bộ `EclatStats`).

In [ ]:
import os
os.chdir(REPO_DIR)
!julia --project=. demo.jl --minsup 40% -o - data/toy/pasquier99.txt
!julia --project=. demo.jl --minsup 0.5 --stats -o output_bookstore.txt data/toy/bookstore.txt

## Phần 4. Kiểm thử tự động ba tầng (`test/runtests.jl`)

Thực thi toàn bộ bộ kiểm thử tự động bao gồm: kiểm thử đọc/ghi chuẩn SPMF và quy đổi ngưỡng `minsup_count`, kiểm chứng bằng bộ duyệt trâu (oracle) trên 200 cơ sở dữ liệu ngẫu nhiên, các tập chuẩn, trường hợp biên, mô-đun luật kết hợp (`rules.jl`) và đối chiếu trên 4 bộ dữ liệu benchmark.

In [ ]:
import os
os.chdir(REPO_DIR)
!julia --project=. test/runtests.jl

## Phần 5. Chương 2 — Thực nghiệm đánh giá hiệu năng

Thực thi hạ tầng đo đạc `scripts/bench.jl` theo giao thức tiến trình độc lập có pha khởi động (warm-up) và xuất kết quả dưới dạng tệp CSV chuẩn 17 cột phục vụ sinh biểu đồ.

In [ ]:
import os
os.chdir(REPO_DIR)
mode_arg = "quick" if CHE_DO == "nhanh" else "full"
repeats = 1 if CHE_DO == "nhanh" else 3
!julia --project=. scripts/bench.jl --mode {mode_arg} --variants v0,v1 --repeats {repeats} --out results/bench_colab.csv
!head -n 20 results/bench_colab.csv

## Phần 6. Chương 3 — Ứng dụng 1: Phân tích giỏ hàng trên tập dữ liệu Groceries

Tái lập toàn bộ quy trình Ứng dụng 1 từ dữ liệu `groceries.csv` (9.835 giao dịch, 169 mặt hàng): so sánh hai phương án biến đổi dữ liệu (Bảng B08), khảo sát độ nhạy lưới ngưỡng, trích xuất 10 luật dẫn đầu theo độ nâng (Bảng B09) và đánh giá độ ổn định trên phép chia 80/20 theo hạt giống cố định (Bảng B10).

In [ ]:
import os
from IPython.display import Markdown, display

os.chdir(REPO_DIR)
!julia --project=. scripts/fetch_data.jl groceries
!julia --project=. apps/groceries/run_groceries.jl

with open("results/groceries_summary.md", "r", encoding="utf-8") as f:
    display(Markdown(f.read()))

## Phần 7. Chương 3 — Ứng dụng 2: Mẫu trực quan bằng túi từ thị giác (Bag of Visual Words)

Nội dung thực thi Ứng dụng 2 trên tập ảnh Caltech-101 được tích hợp sau khi hoàn thiện mô-đun `apps/bovw/`.

## Phần 8. Tổng hợp tài liệu và báo cáo

Báo cáo hoàn chỉnh theo mẫu hội nghị IEEE được lưu trữ tại `docs/Report.pdf` trong kho mã nguồn của nhóm.